In [1]:
# ======================================
# PHASE 4 — RADAR HYBRID MODEL
# ======================================

import numpy as np
import pandas as pd
import os

#from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [2]:
#cell 2 

X_train = np.load("/Users/amarachi/Documents/AQA/Exp_3/phase3/notebook /phase3_features/X_train.npy")
X_val = np.load("/Users/amarachi/Documents/AQA/Exp_3/phase3/notebook /phase3_features/X_val.npy")
X_test = np.load("/Users/amarachi/Documents/AQA/Exp_3/phase3/notebook /phase3_features/X_test.npy")

y_train = np.load("/Users/amarachi/Documents/AQA/Exp_3/phase3/notebook /phase3_features/y_train.npy")
y_val = np.load("/Users/amarachi/Documents/AQA/Exp_3/phase3/notebook /phase3_features/y_val.npy")
y_test = np.load("/Users/amarachi/Documents/AQA/Exp_3/phase3/notebook /phase3_features/y_test.npy")

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (2338, 776)
Validation: (621, 776)
Test: (355, 776)


In [3]:
#cell 3
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

base_model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model = CalibratedClassifierCV(base_model, method="isotonic", cv=3)

model.fit(X_train, y_train)

print("Calibrated Gradient Boosting model trained.")

Calibrated Gradient Boosting model trained.


In [4]:
#cell 4

train_probs = model.predict_proba(X_train)[:,1]
val_probs = model.predict_proba(X_val)[:,1]
test_probs = model.predict_proba(X_test)[:,1]

In [5]:
#cell 5

best_score = 0
best_threshold = 0.5

for t in np.linspace(0.05,0.95,181):

    preds = (val_probs >= t).astype(int)

    precision = precision_score(y_val, preds)
    recall = recall_score(y_val, preds)
    f1 = f1_score(y_val, preds)

    # weighted objective
    score = (2 * precision + recall) / 3

    if score > best_score:
        best_score = score
        best_threshold = t

print("Best threshold:", best_threshold)
print("Validation score:", best_score)

Best threshold: 0.59
Validation score: 0.9690745575863069


In [6]:
#cell 5

In [7]:
#cell 6 

test_preds = (test_probs >= best_threshold).astype(int)

precision = precision_score(y_test,test_preds)
recall = recall_score(y_test,test_preds)
f1 = f1_score(y_test,test_preds)
auc = roc_auc_score(y_test,test_probs)

print("Precision:",precision)
print("Recall:",recall)
print("F1:",f1)
print("AUC:",auc)

Precision: 0.8798076923076923
Recall: 0.8133333333333334
F1: 0.8452655889145496
AUC: 0.8266153846153846


In [8]:
#cell 7

cm = confusion_matrix(y_test,test_preds)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[105  25]
 [ 42 183]]


In [9]:
#cell 8

# ===============================
# RADAR Risk-Aware Decision Rule
# ===============================

from sklearn.metrics import precision_score, recall_score, confusion_matrix
import numpy as np

CFN = 5
CFP = 1

tau = CFP / (CFP + CFN)

print("Theoretical risk threshold:", tau)

# Adjust threshold relative to validation distribution
risk_threshold = np.quantile(val_probs, tau)

print("Adjusted risk threshold:", risk_threshold)

risk_preds = (test_probs >= risk_threshold).astype(int)

print("\nRisk-aware Precision:", precision_score(y_test, risk_preds))
print("Risk-aware Recall:", recall_score(y_test, risk_preds))

print("\nRisk-aware confusion matrix:")
print(confusion_matrix(y_test, risk_preds))

Theoretical risk threshold: 0.16666666666666666
Adjusted risk threshold: 0.43481223015431975

Risk-aware Precision: 0.6422287390029325
Risk-aware Recall: 0.9733333333333334

Risk-aware confusion matrix:
[[  8 122]
 [  6 219]]


In [10]:
import os
import numpy as np

os.makedirs("phase4_outputs", exist_ok=True)

np.save("phase4_outputs/test_probs.npy", test_probs)
np.save("phase4_outputs/test_preds.npy", test_preds)

print("Phase 4 outputs saved.")

Phase 4 outputs saved.


In [11]:
print("Default threshold confusion matrix:")
print(confusion_matrix(y_test, test_preds))

print("\nRisk-aware confusion matrix:")
print(confusion_matrix(y_test, risk_preds))

Default threshold confusion matrix:
[[105  25]
 [ 42 183]]

Risk-aware confusion matrix:
[[  8 122]
 [  6 219]]


In [12]:
#cell 9

fn_default = confusion_matrix(y_test,test_preds)[1][0]
fn_risk = confusion_matrix(y_test,risk_preds)[1][0]

results = {
    "precision":precision,
    "recall":recall,
    "f1":f1,
    "auc":auc,
    "threshold":best_threshold,
    "risk_threshold":tau,
    "FN_default":fn_default,
    "FN_risk":fn_risk
}

pd.DataFrame([results]).to_csv("radar_results.csv",index=False)

print("Results saved.")

Results saved.


In [13]:
#cell 10

print("Train probability range:",train_probs.min(),train_probs.max())
print("Test probability range:",test_probs.min(),test_probs.max())

Train probability range: 0.17761557177615572 1.0
Test probability range: 0.27898462766798543 1.0


In [14]:
#cell 11

In [15]:
#cell 12

